In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import false_discovery_control
#from bokeh.models import FixedTicker, FuncTickFormatter 
from scipy.stats import chi2

#from bokeh.io import output_notebook

from snp_analysis_tools_sherlock import *
from scipy.stats import binom
import iqplot
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot
from glob import glob

hv.extension('bokeh')

In [ ]:
import os
inos=['AE-AF-mBHI','AA-AF-mBHI','AA-AE-mBHI', 'AE-AF-mGAM','AA-AF-mGAM','AA-AE-mGAM']
#inos = ['AA-AF-mGAM']
#inos = ['AE-AF-mBHI']
species = [102438,100196, 100099, 100146, 101346, 102506, 10258,102478, 102506,102528]
#species =  [102478,100146]
alls = []
for sp in species:
    plots_all=[]
    for ino in inos:
        fname1 = f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/track_snpsv2_ALL_bootstrapv3/{sp}/{ino}_parent1_info.csv'
        
        if not os.path.isfile(fname1):
            continue
        df1 = pd.read_csv(fname1).drop(columns='Unnamed: 0')
        df1 = df1.rename(columns={'actual_med':'actual_mednew'})
        df1 = df1.set_index('sample')
        fname2 = f'/Users/sophiewalton/git/coalescence-pilot-mgx/workflow/report/old_bs/track_snpsv2_ALL_bootstrapv3/{sp}/{ino}_parent1_info.csv'
        df2= pd.read_csv(fname2).drop(columns='Unnamed: 0')
        df2 = df2.rename(columns={'actual_med':'actual_medold'})
        df2 = df2.set_index('sample')
        
        both=pd.concat([df1,df2],axis=1)
       # print(both.head())
        both['species_id']=str(sp)
        alls.append(both)
      # print(both)
   
   # !rsvg-convert -f pdf -o {f'{test_name}.pdf'} {f'{test_name}.svg'} 
    
#,filename=f'{sp}_linear_preds_12.png'
    #bokeh.io.export_png(bokeh.layouts.gridplot(plots_all,ncols=4),filename=f'{sp}_linear_preds_12.png')

In [ ]:
all_df = pd.concat(alls)
all_df.loc[all_df['actual_mednew']<5e-3,'actual_mednew']=5e-3
all_df.loc[all_df['actual_mednew']>1-5e-3,'actual_mednew']=1-5e-3

all_df.loc[all_df['actual_medold']<5e-3,'actual_medold']=5e-3
all_df.loc[all_df['actual_medold']>1-5e-3,'actual_medold']=1-5e-3
all_df['actual_mednew_logit']=np.log(all_df['actual_mednew']/(1-all_df['actual_mednew']))

all_df['actual_medold_logit']=np.log(all_df['actual_medold']/(1-all_df['actual_medold']))

In [ ]:
scat = hv.Scatter(all_df, vdims = ['actual_mednew'],kdims=['actual_medold']).opts(width=400,alpha = .2)
scat

In [ ]:
b=all_df.loc[all_df['actual_mednew']==1-1e-3,:]
print(len(b))
len(b.loc[b['actual_medold']<1-1e-3,:])

In [ ]:
scat = hv.Scatter(all_df, vdims = ['actual_mednew_logit'],kdims=['actual_medold_logit']).opts(width=400,alpha = .2)

exps_adjust = np.array([.001,.01,.1,.5,.9,.99,.999])
    
lins_adjust = np.log(exps_adjust/(1-exps_adjust))
scat.opts(xticks=[(lins_adjust[i], exps_adjust[i]) for i in range(len(lins_adjust))],
                 yticks=[(lins_adjust[i], exps_adjust[i]) for i in range(len(lins_adjust))]
                )